# Export completo dei ranking cane–famiglia con preferenze di razza

Questo notebook calcola il ranking completo dei cani per uno specifico profilo famiglia e salva il risultato in un file CSV.

Rispetto alla versione precedente, il sistema considera anche le **razze preferite** indicate dalla famiglia.

Il CSV finale contiene tutti i cani del dataset, ordinati per `final_score`, e mette all'inizio della riga i punteggi principali:

- `final_score`;
- `structured_score`;
- `breed_score`;
- `bert_similarity`.

In questo modo il file è più leggibile quando viene aperto in Excel.


## 1. Import delle librerie

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- `dogs_clean.csv`;
- `bert_embeddings.npy`;
- `family_profiles.json`.

Se il dataset non contiene ancora le colonne `breed1_label` e `breed2_label`, queste vengono create automaticamente usando `breed_labels.csv`.


In [2]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

# Creazione automatica delle etichette di razza se non presenti
if "breed1_label" not in dogs.columns or "breed2_label" not in dogs.columns:
    breed_labels = pd.read_csv("../data/raw/breed_labels.csv")

    breed_map = dict(
        zip(
            breed_labels["BreedID"],
            breed_labels["BreedName"]
        )
    )

    dogs["breed1_label"] = dogs["Breed1"].map(breed_map)
    dogs["breed2_label"] = dogs["Breed2"].map(breed_map)
    dogs["breed2_label"] = dogs["breed2_label"].fillna("None")

profiles_path = Path("../data/family_profiles.json")

with open(profiles_path, "r", encoding="utf-8") as f:
    family_profiles = json.load(f)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Profili famiglia disponibili:")
for profile_id in family_profiles.keys():
    print("-", profile_id)

Dataset cani: (8132, 32)
Embedding BERT: (8132, 768)
Profili famiglia disponibili:
- family_children_apartment
- single_active_person
- senior_applicant
- experienced_countryside_family
- first_time_adopter


## 3. Scelta del profilo famiglia

Modifica `selected_profile_id` per scegliere il profilo su cui calcolare il ranking.

Profili disponibili, ad esempio:

- `family_children_apartment`
- `single_active_person`
- `senior_applicant`
- `experienced_countryside_family`
- `first_time_adopter`


In [3]:
selected_profile_id = "family_children_apartment"

family_profile = family_profiles[selected_profile_id]

family_profile

{'profile_name': 'Famiglia con bambini in appartamento',
 'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'apartment',
 'garden': False,
 'preferred_gender': 'indifferent',
 'preferred_age': 'young',
 'preferred_size': 'small',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'preferred_breeds': ['Labrador Retriever', 'Golden Retriever'],
 'ideal_dog_description': 'We are looking for a small, affectionate and friendly dog suitable for a family with children. The dog should be calm at home, sociable, easy to manage and suitable for apartment life.'}

## 4. Funzioni di compatibilità con razza

La razza viene gestita con un punteggio specifico:

- razza principale compatibile: `breed_score = 1.0`;
- seconda razza compatibile: `breed_score = 0.6`;
- nessuna corrispondenza: `breed_score = 0.0`;
- nessuna preferenza di razza: `breed_score = 1.0`.

Nel calcolo strutturato la razza ha peso 3.


In [4]:
def normalize_text(text):
    return str(text).strip().lower()


def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_breed_score(dog, family):
    preferred_breeds = family.get("preferred_breeds", [])

    if len(preferred_breeds) == 0:
        return 1.0

    preferred_breeds = [
        normalize_text(breed)
        for breed in preferred_breeds
    ]

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    if breed1 in preferred_breeds:
        return 1.0

    if breed2 in preferred_breeds:
        return 0.6

    return 0.0


def compute_structured_score(dog, family):
    # Vincolo hard: meno di 1 ora al giorno non è compatibile con l'adozione.
    if family["daily_time"] == "<1":
        return 0.0

    # Vincolo hard: cani extra large richiedono un giardino.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # Vincolo hard: nessuna esperienza + cane extra large.
    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    weighted_scores = []

    gender_score = match_preference(family["preferred_gender"], dog["gender_label"])
    age_score = match_preference(family["preferred_age"], dog["age_group"])
    size_score = size_similarity(family["preferred_size"], dog["maturity_size_label"])
    fur_score = match_preference(family["preferred_fur"], dog["fur_length_label"])
    breed_score = compute_breed_score(dog, family)

    weighted_scores.append((gender_score, 1))
    weighted_scores.append((age_score, 1))
    weighted_scores.append((size_score, 1))
    weighted_scores.append((fur_score, 1))

    # La razza ha peso maggiore
    weighted_scores.append((breed_score, 3))

    # Regola bambini
    if family["children"]:
        children_score = 1.0 if dog["age_group"] in ["puppy", "young"] else 0.5
        weighted_scores.append((children_score, 1))

    # Regola giardino e taglia
    if dog["maturity_size_label"] == "large":
        garden_score = 1.0 if family["garden"] else 0.2
        weighted_scores.append((garden_score, 1))
    elif dog["maturity_size_label"] == "extra_large":
        weighted_scores.append((1.0, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola tempo disponibile
    if family["daily_time"] == "1-2":
        time_score = 0.6 if dog["age_group"] == "puppy" else 0.8
        weighted_scores.append((time_score, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola esperienza
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            experience_score = 0.2
        elif dog["age_group"] == "puppy":
            experience_score = 0.5
        else:
            experience_score = 1.0
    else:
        experience_score = 1.0

    weighted_scores.append((experience_score, 1))

    # Regola over 60
    if family["applicant_age"] == "over_60":
        senior_score = 1.0 if dog["age_group"] in ["adult", "senior"] else 0.4
        weighted_scores.append((senior_score, 1))

    total_score = sum(score * weight for score, weight in weighted_scores)
    total_weight = sum(weight for score, weight in weighted_scores)

    return total_score / total_weight

## 5. Calcolo dello score strutturato

La funzione viene applicata a tutti i cani del dataset.


In [5]:
dogs_export = dogs.copy()

dogs_export["breed_score"] = dogs_export.apply(
    lambda row: compute_breed_score(row, family_profile),
    axis=1
)

dogs_export["structured_score"] = dogs_export.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

print("Numero totale cani:", len(dogs_export))
print("Cani esclusi dai vincoli hard:", (dogs_export["structured_score"] == 0).sum())
print("Cani valutati:", (dogs_export["structured_score"] > 0).sum())

dogs_export[[
    "Name",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "structured_score"
]].head()

Numero totale cani: 8132
Cani esclusi dai vincoli hard: 22
Cani valutati: 8110


,Name,breed1_label,breed2_label,breed_score,structured_score
0,Brisco,Mixed Breed,NaN,0.0,0.500000
1,Miko,Mixed Breed,NaN,0.0,0.590909
2,Hunter,Mixed Breed,NaN,0.0,0.590909
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,0.0,0.590909
4,Bear,Mixed Breed,NaN,0.0,0.590909


## 6. Similarità semantica tramite BERT

Il testo libero della famiglia viene trasformato in embedding BERT e confrontato con gli embedding delle descrizioni dei cani.


In [6]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [7]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

dogs_export["bert_similarity"] = (semantic_scores + 1) / 2

dogs_export[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907398
1,Miko,0.876957
2,Hunter,0.925550
3,Siu Pak & Her 6 Puppies,0.822069
4,Bear,0.827598


## 7. Calcolo dello score finale

In [8]:
dogs_export["final_score"] = (
    0.7 * dogs_export["structured_score"] +
    0.3 * dogs_export["bert_similarity"]
)

dogs_export[[
    "Name",
    "final_score",
    "structured_score",
    "breed_score",
    "bert_similarity"
]].head()

,Name,final_score,structured_score,breed_score,bert_similarity
0,Brisco,0.622220,0.500000,0.0,0.907398
1,Miko,0.676724,0.590909,0.0,0.876957
2,Hunter,0.691301,0.590909,0.0,0.925550
3,Siu Pak & Her 6 Puppies,0.660257,0.590909,0.0,0.822069
4,Bear,0.661916,0.590909,0.0,0.827598


## 8. Creazione del ranking completo

Il ranking contiene tutti i cani del dataset, ordinati per `final_score`.

I punteggi vengono messi all'inizio delle colonne del CSV.


In [9]:
export_columns = [
    "final_score",
    "structured_score",
    "breed_score",
    "bert_similarity",

    "PetID",
    "Name",
    "breed1_label",
    "breed2_label",
    "Age",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
    "Fee",
    "PhotoAmt",
    "Description"
]

full_ranking = dogs_export.sort_values(
    by="final_score",
    ascending=False
)[export_columns]

full_ranking = full_ranking.round({
    "final_score": 4,
    "structured_score": 4,
    "breed_score": 4,
    "bert_similarity": 4
})

print("Numero cani nel ranking completo:", len(full_ranking))

full_ranking.head(20)

Numero cani nel ranking completo: 8132


,final_score,structured_score,breed_score,bert_similarity,PetID,Name,breed1_label,breed2_label,Age,age_group,gender_label,maturity_size_label,fur_length_label,Vaccinated,Dewormed,Sterilized,Health,Fee,PhotoAmt,Description
5743,0.9654,1.0000,1.0,0.8848,01c6d3224,Bobby,Golden Retriever,NaN,8,young,male,small,short,1,1,2,1,600,5.0,Golden retriever 8months old.Need a loving hom...
8089,0.9415,0.9545,1.0,0.9111,b3a6f353d,Pacino,Labrador Retriever,Mixed Breed,12,young,male,medium,short,3,1,1,1,0,7.0,PAcino was found at a park in Damansara Kim on...
7580,0.9390,0.9545,1.0,0.9026,295dac1ce,"Mummy Moon, Luna And Ariel",Golden Retriever,Labrador Retriever,15,young,female,medium,short,3,1,3,1,0,3.0,Three pups were rescued during Thaipusam and f...
3697,0.9388,0.9545,1.0,0.9019,ab9dd85b2,2 Black Dude,Labrador Retriever,Mixed Breed,8,young,mixed,medium,short,1,1,2,1,0,7.0,"siblings brother and sister, big paws and shin..."
2549,0.9379,0.9545,1.0,0.8990,e246ecfdc,FLUFFY,Golden Retriever,NaN,22,young,female,medium,short,1,1,2,1,0,2.0,i am the owner for this dog and i am looking f...
1285,0.9375,0.9545,1.0,0.8977,902114441,Whiskey,Labrador Retriever,Mixed Breed,12,young,male,medium,short,3,3,3,1,0,2.0,Whiskey is our own pet. An offspring of my aun...
1898,0.9274,0.9545,1.0,0.8642,2d88536fc,Nil,Labrador Retriever,Mixed Breed,12,young,male,medium,short,2,1,1,1,30,1.0,I'm a labrador retriever cross local. I have t...
3309,0.9105,0.9091,1.0,0.9137,9430d84c5,Snow,Labrador Retriever,Mixed Breed,12,young,female,small,long,1,1,1,1,0,4.0,Snow used to have a home nearby my area but wh...
37,0.9047,0.9091,1.0,0.8945,1e62e6022,Baby / Bubbles,Labrador Retriever,Jack Russell Terrier,2,puppy,male,small,short,1,3,2,1,0,5.0,3 month-old male puppy for adoption. a cross b...
2757,0.8879,0.9091,1.0,0.8385,a64d70c84,Simba,Labrador Retriever,Beagle,2,puppy,female,small,short,1,1,2,1,0,0.0,Simba is very loving. She is also very playful...


## 9. Esportazione in CSV

Il file CSV viene salvato nella cartella `data/results`.

Il nome del file contiene l'ID del profilo famiglia scelto.


In [10]:
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / f"ranking_{selected_profile_id}.csv"

full_ranking.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("File CSV salvato in:")
print(output_file)

print("Numero cani esportati:")
print(len(full_ranking))

File CSV salvato in:
..\data\results\ranking_family_children_apartment.csv
Numero cani esportati:
8132


## 10. Esportazione opzionale della Top 20

In [11]:
top20_file = results_dir / f"top20_{selected_profile_id}.csv"

full_ranking.head(20).to_csv(
    top20_file,
    index=False,
    encoding="utf-8"
)

print("File Top 20 salvato in:")
print(top20_file)

File Top 20 salvato in:
..\data\results\top20_family_children_apartment.csv


## 11. Conclusioni

Questo notebook esporta un ranking completo per uno specifico profilo famiglia.

Il file CSV prodotto contiene tutti i cani valutati dal sistema e permette di analizzare:

- score finale;
- compatibilità strutturata;
- compatibilità di razza;
- similarità BERT;
- caratteristiche principali del cane.

L'aggiunta del `breed_score` rende il ranking più coerente con le preferenze esplicite della famiglia.
